# Nhóm 13 · M1 — Cào dữ liệu trên Colab CPU

**Chỉ chọn MSSV ở cell 1 rồi Runtime → Run all.** Lần đầu Google vẫn có thể hỏi
quyền Secrets/Drive; không thể bỏ qua các hộp cấp quyền này bằng mã.

- Mỗi người dùng key S2 cá nhân, bật Notebook access cho hai Secrets và chỉ chạy
  **một runtime cho MSSV của mình**. Ba người dùng cùng source và thư mục Drive.
- Mặc định chỉ cào 5.000 ID của MSSV đã chọn; probe 5 bài tự chuyển sang full khi đạt.
- Worker chạy nền, không bị dừng chỉ vì cell Monitor kết thúc; không dùng cell
  "giữ máy sống". Google vẫn có thể thu hồi runtime. Chạy lại để resume.
- Bảng thời gian gồm cache/import, metadata, source, references, ngày references,
  kiểm tra và ghi Drive; mean/p50/p95 tính theo lượt xử lý, không giả làm cold benchmark.
- Output ổn định: `m1-team-v4/<MSSV>/main/full/<owner>/`; pilot dùng ngay dữ liệu này,
  không tạo corpus pilot riêng. File tạm ở SSD local; checkpoint/log benchmark được giữ.
- arXiv chia lượt UTC giữa ba MSSV để không cộng dồn quota. Chạy riêng muốn bỏ
  thời gian chờ phải xác nhận các crawler khác đã dừng qua script, không tự suy đoán.

Notebook chỉ cào/hiển thị. Parser, matching, xuất bản và audit chạy bằng script Python.


In [304]:
# @title 1. Chọn MSSV · Chuẩn bị Colab và mã nguồn
STUDENT_ID = "23120364" # @param ["23120364", "23120358", "23120345"]
import os, sys, hashlib, zipfile, subprocess
from pathlib import Path
from google.colab import userdata, drive

assert os.environ.get("COLAB_GPU", "0") in ("", "0"), "Chọn runtime CPU, không GPU."
for name in ("ARXIV_CONTACT_EMAIL", "SEMANTIC_SCHOLAR_API_KEY"):
    os.environ[name] = userdata.get(name)
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/IDS_Nhom13_Corpus_Backup")
assert DRIVE_ROOT.is_dir(), "Thêm shortcut thư mục Drive chung vào My Drive với đúng tên."
SOURCE_SHA256 = "ab50c956abf90c5c83b5dd75065cfb1c767f707b6f5fd95d18236639ee240889"
archive = DRIVE_ROOT / ("ids13-code-" + SOURCE_SHA256[:12] + ".zip")
assert hashlib.sha256(archive.read_bytes()).hexdigest() == SOURCE_SHA256, "Source checksum mismatch"
CODE_ROOT = Path("/content") / ("ids13-code-" + SOURCE_SHA256[:12])
CODE_ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as bundle:
    for item in bundle.infolist():
        assert (CODE_ROOT/item.filename).resolve().is_relative_to(CODE_ROOT.resolve()), "Unsafe archive path"
        assert (item.external_attr >> 16) & 0o170000 != 0o120000, "Archive symlink"
    bundle.extractall(CODE_ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_ROOT/"requirements-colab-m1.txt")], check=True)
os.environ["PYTHONPATH"] = str(CODE_ROOT/"src")
os.environ["IDS_SOURCE_SHA256"] = SOURCE_SHA256
print("Sẵn sàng:", STUDENT_ID, "| source", SOURCE_SHA256[:12], "| không hiển thị Secrets")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sẵn sàng: 23120364 | source 179d7935569c | không hiển thị Secrets


In [305]:
# @title 2. Tự kiểm tra → probe 5 bài → cào đủ dải (chạy nền)
import importlib.util
spec = importlib.util.spec_from_file_location("_ids13_colab_entry", CODE_ROOT/"scripts/colab_entry.py")
entry = importlib.util.module_from_spec(spec)
spec.loader.exec_module(entry)
APP = entry.start(STUDENT_ID, DRIVE_ROOT, CODE_ROOT)


Tests: đạt. Preflight: {'runner_id': '23120364', 'targets': {'23120364': 5000}, 'protocol_version': 4, 'source_sha256': '179d7935569cb4e6832ee0b2dd6f496b1ea757d4542f309c77aa5407cabb5c0f', 'packages': {'requests': '2.32.4', 'pandas': '2.2.3', 'pyarrow': '23.0.1', 'pydantic': '2.14.2', 'psutil': '5.9.5', 'pytest': '8.4.2'}}
Khởi chạy probe 5 bài → tự chuyển đủ dải khi probe đạt: {'launched': 161754, 'run_id': '60e6cbe4472d429a945272c422becf6a', 'runner_id': '23120364', 'target_owner_ids': ['23120364'], 'stage': 'auto'}


In [310]:
# @title 3. Tiến độ · Tốc độ từng khâu · Độ trễ mỗi lượt paper
APP.monitor(seconds=30)
# Xem tiếp: chạy lại cell này. Dừng có kiểm soát (không chạy mặc định):
# APP.stop(confirm_run_id="copy chính xác run_id từ trạng thái")


[22:18:43] RUNNING | khâu: paper | paper: 2508.18614
  23120364: đạt 54/5000 | source 2154 | refs 152
  Độ trễ/lượt: mean 0.60s | p50 0.36s | p95 2.57s (n=394)
  Khâu                             n    mean(s)    p95(s)
  cache_check_import              394      0.104     0.308
  s2_prefetch                     394      0.000     0.001
  paper_worker                    394      0.183     0.767
  drive_commit                    394      0.250     0.693
  checkpoint                      394      0.064     0.104
  worker/overhead                 394      0.005     0.007
  worker/metadata                 394      0.014     0.028
  worker/references               394      0.117     0.465
  worker/reference_dates          394      0.028     0.145
  worker/validate_local           394      0.009     0.013
  worker/staged                   394      0.004     0.006
  metadata_batch_shared             9      0.005     0.006
[22:18:53] RUNNING | khâu: commit | paper: 2508.20049
  23120364: đạt 54/5